In [29]:
from pathlib import Path
from io import BytesIO
import re
import time

import pandas as pd
import requests
from PIL import Image, ImageOps, ImageEnhance
import pytesseract
from IPython.display import display

RAW_DIR = Path("../raw")
INTERIM_DIR = Path("../interim")

CANDIDATE_PATH = (
    RAW_DIR / "lok_sabha_candidate_dfs" / "candidates_all_years.csv"
)
LINK_PATH = (
    RAW_DIR / "lok_sabha_constituencies"
    / "constituency_links_all_years.csv"
)

OCR_CHECKPOINT_PATH = INTERIM_DIR / "image_ocr_checkpoint.csv"
OUTPUT_PATH = INTERIM_DIR / "candidates_processed.csv"
AUDIT_PATH = INTERIM_DIR / "candidate_processing_audit.csv"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 20
REQUEST_TIMEOUT = (10, 30)
REQUEST_DELAY = 0.15

print("Working directory:", Path.cwd())
print("Interim directory:", INTERIM_DIR.resolve())
print("Candidate input:", CANDIDATE_PATH.resolve())
print("Constituency reference:", LINK_PATH.resolve())

Working directory: d:\MyCoding\DA2402\DCV-Project\notebooks
Interim directory: D:\MyCoding\DA2402\DCV-Project\interim
Candidate input: D:\MyCoding\DA2402\DCV-Project\raw\lok_sabha_candidate_dfs\candidates_all_years.csv
Constituency reference: D:\MyCoding\DA2402\DCV-Project\raw\lok_sabha_constituencies\constituency_links_all_years.csv


In [30]:
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

print("Tesseract version:", pytesseract.get_tesseract_version())

Tesseract version: 5.4.0.20240606


Test Run

In [31]:
import requests
import pytesseract
from PIL import Image, ImageOps, ImageEnhance
from io import BytesIO

# Tesseract executable (Windows)
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

# Image URL you provided
url = "https://myneta.info/image_v2.php?myneta_folder=LokSabha2024&candidate_id=3627&col=ta"

try:
    # 1. Download image into memory (no file created)
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    # 2. Load and preprocess image
    image = Image.open(BytesIO(response.content))
    image.load()
    image = ImageOps.grayscale(image)
    image = ImageEnhance.Contrast(image).enhance(2)

    # 3. Extract all visible text
    text = pytesseract.image_to_string(
        image,
        config="--oem 3 --psm 6"
    )

    print("OCR TEXT:")
    print(text if text.strip() else "[No text detected]")

    # 4. Release image data from memory
    image.close()
    del image, response

except Exception as e:
    print(f"Failed to download or process image: {e}")

OCR TEXT:
Rs 1,60,59,806
~1 Crore+.



In [32]:
candidates = pd.read_csv(CANDIDATE_PATH)
links = pd.read_csv(LINK_PATH)

print("Candidates shape:", candidates.shape)
print("Constituency links shape:", links.shape)

print("\nCandidate columns:")
print(candidates.columns.tolist())

print("\nConstituency-link columns:")
print(links.columns.tolist())

display(candidates.head())
display(links.head())

Candidates shape: (10395, 23)
Constituency links shape: (1679, 10)

Candidate columns:
['year', 'state', 'state_id', 'const', 'const_id', 'sno', 'candidate', 'candidate_id', 'candidate_url', 'is_winner', 'party', 'criminal_cases', 'education', 'age', 'total_assets', 'asset_link', 'liabilities', 'liability_link', 'constituency_url', 'html_path', 'is_byelection', 'bye_election_date', 'election_date']

Constituency-link columns:
['year', 'state_ut', 'state_id', 'const', 'const_id', 'link', 'anchor_title', 'election_date', 'is_byelection', 'bye_election_date']


,year,state,state_id,const,const_id,sno,candidate,candidate_id,candidate_url,is_winner,...,age,total_assets,asset_link,liabilities,liability_link,constituency_url,html_path,is_byelection,bye_election_date,election_date
0,2024,BIHAR,5,ARARIA,56,6,Pradeep Kumar Singh,3627,https://www.myneta.info/LokSabha2024/candidate...,True,...,58,NaN,https://myneta.info/image_v2.php?myneta_folder...,NaN,https://myneta.info/image_v2.php?myneta_folder...,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html,False,NaN,07-05-2024
1,2024,BIHAR,5,ARARIA,56,1,Akhilesh Kumar,4040,https://www.myneta.info/LokSabha2024/candidate...,False,...,47,"Rs 2,47,11,288 ~ 2 Crore+",NaN,"Rs 3,05,374 ~ 3 Lacs+",NaN,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html,False,NaN,07-05-2024
2,2024,BIHAR,5,ARARIA,56,2,Md Ghousul Azam,4773,https://www.myneta.info/LokSabha2024/candidate...,False,...,38,"Rs 1,60,50,000 ~ 1 Crore+",NaN,"Rs 14,00,000 ~ 14 Lacs+",NaN,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html,False,NaN,07-05-2024
3,2024,BIHAR,5,ARARIA,56,3,Md. Ismail,4107,https://www.myneta.info/LokSabha2024/candidate...,False,...,56,NaN,https://myneta.info/image_v2.php?myneta_folder...,NaN,https://myneta.info/image_v2.php?myneta_folder...,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html,False,NaN,07-05-2024
4,2024,BIHAR,5,ARARIA,56,4,Md. Mobinul Haque,4772,https://www.myneta.info/LokSabha2024/candidate...,False,...,51,"Rs 34,05,000 ~ 34 Lacs+",NaN,Rs 0 ~,NaN,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html,False,NaN,07-05-2024


,year,state_ut,state_id,const,const_id,link,anchor_title,election_date,is_byelection,bye_election_date
0,2024,ANDAMAN AND NICOBAR ISLANDS,1,ANDAMAN AND NICOBAR ISLANDS,579,https://www.myneta.info/LokSabha2024/index.php...,Date of Election 19-04-2024,19-04-2024,False,NaN
1,2024,ANDHRA PRADESH,2,AMALAPURAM (SC),7,https://www.myneta.info/LokSabha2024/index.php...,Date of Election 13-05-2024,13-05-2024,False,NaN
2,2024,ANDHRA PRADESH,2,ANAKAPALLE,5,https://www.myneta.info/LokSabha2024/index.php...,Date of Election 13-05-2024,13-05-2024,False,NaN
3,2024,ANDHRA PRADESH,2,ANANTHAPUR,20,https://www.myneta.info/LokSabha2024/index.php...,Date of Election 13-05-2024,13-05-2024,False,NaN
4,2024,ANDHRA PRADESH,2,ARAKU (ST),1,https://www.myneta.info/LokSabha2024/index.php...,Date of Election 13-05-2024,13-05-2024,False,NaN


In [33]:
required_candidate_cols = [
    "year", "state", "state_id", "const", "const_id",
    "sno", "candidate", "candidate_id", "candidate_url",
    "is_winner", "party", "criminal_cases", "education",
    "age", "total_assets", "asset_link", "liabilities",
    "liability_link", "constituency_url", "html_path",
    "is_byelection", "bye_election_date", "election_date"
]

required_link_cols = [
    "year", "state_ut", "state_id", "const",
    "const_id", "link", "election_date",
    "is_byelection", "bye_election_date"
]

missing_candidate_cols = sorted(
    set(required_candidate_cols) - set(candidates.columns)
)
missing_link_cols = sorted(
    set(required_link_cols) - set(links.columns)
)

print("Missing candidate columns:", missing_candidate_cols)
print("Missing link columns:", missing_link_cols)

if missing_candidate_cols or missing_link_cols:
    raise ValueError("Fix the schema mismatches before continuing.")

print("\nCandidate dtypes:")
display(candidates.dtypes.astype(str).to_frame("dtype"))

print("\nMissing values in candidate data:")
display(candidates.isna().sum().to_frame("missing_count"))

Missing candidate columns: []
Missing link columns: []

Candidate dtypes:


,dtype
year,int64
state,str
state_id,int64
const,str
const_id,int64
sno,int64
candidate,str
candidate_id,int64
candidate_url,str
is_winner,bool



Missing values in candidate data:


,missing_count
year,0
state,0
state_id,0
const,0
const_id,0
sno,0
candidate,0
candidate_id,0
candidate_url,0
is_winner,0


In [34]:
print("Exact duplicate candidate rows:", candidates.duplicated().sum())

candidate_key = ["year", "state", "const", "sno"]

print(
    "Duplicate candidate-event keys:",
    candidates.duplicated(candidate_key, keep=False).sum()
)

display(
    candidates.loc[
        candidates.duplicated(candidate_key, keep=False),
        candidate_key + ["candidate", "candidate_id"]
    ].sort_values(candidate_key).head(30)
)

print("\nYear counts:")
display(candidates["year"].value_counts(dropna=False).sort_index())

print("\nWinner values:")
display(candidates["is_winner"].value_counts(dropna=False))

print("\nBye-election values:")
display(candidates["is_byelection"].value_counts(dropna=False))

Exact duplicate candidate rows: 0
Duplicate candidate-event keys: 268


,year,state,const,sno,candidate,candidate_id
6686,2014,BIHAR,ARARIA,1,Abdul Rahman,5236
6699,2014,BIHAR,ARARIA,1,Binit Prakash,10100
6687,2014,BIHAR,ARARIA,2,Bidya Nand Paswan,4453
6700,2014,BIHAR,ARARIA,2,Maheshwar Rishi,10101
6688,2014,BIHAR,ARARIA,3,Chandra Bhushan,4454
6701,2014,BIHAR,ARARIA,3,Pradeep Kumar Singh,10102
6689,2014,BIHAR,ARARIA,4,Md Aslam Beg,5233
6702,2014,BIHAR,ARARIA,4,Prince Victor,10103
6690,2014,BIHAR,ARARIA,5,Pankaj Kishor Mandal,4103
6698,2014,BIHAR,ARARIA,5,Sarfaraz Alam,10098



Year counts:


year
2014    3710
2019    3282
2024    3403
Name: count, dtype: int64


Winner values:


is_winner
False    9761
True      634
Name: count, dtype: int64


Bye-election values:


is_byelection
False    10228
True       167
Name: count, dtype: int64

In [35]:
def clean_key(series):
    return series.astype("string").str.strip().str.casefold()

def clean_url(series):
    return (
        series.astype("string")
        .str.strip()
        .str.rstrip("/")
        .replace("", pd.NA)
    )

candidate_check = candidates.copy()
link_check = links.copy()

candidate_check["_url_key"] = clean_url(
    candidate_check["constituency_url"]
)
link_check["_url_key"] = clean_url(link_check["link"])

candidate_check["_year_key"] = clean_key(candidate_check["year"])
link_check["_year_key"] = clean_key(link_check["year"])

candidate_check["_state_key"] = clean_key(candidate_check["state"])
link_check["_state_key"] = clean_key(link_check["state_ut"])

candidate_check["_const_key"] = clean_key(candidate_check["const"])
link_check["_const_key"] = clean_key(link_check["const"])

print(
    "Duplicate reference URL/year keys:",
    link_check.duplicated(["_url_key", "_year_key"], keep=False).sum()
)

# Preserve the reference data and avoid multiplying candidate rows.
reference = link_check.drop_duplicates(
    ["_url_key", "_year_key"]
)[[
    "_url_key", "_year_key",
    "_state_key", "_const_key",
    "state_ut", "const", "link"
]].rename(columns={
    "_state_key": "_ref_state_key",
    "_const_key": "_ref_const_key",
    "state_ut": "ref_state",
    "const": "ref_const",
    "link": "ref_link"
})

checked = candidate_check.merge(
    reference,
    on=["_url_key", "_year_key"],
    how="left",
    indicator="_url_match",
    validate="many_to_one"
)

checked["url_match"] = checked["_url_match"].eq("both")

checked["state_match"] = (
    checked["_state_key"] == checked["_ref_state_key"]
) & checked["url_match"]

checked["const_match"] = (
    checked["_const_key"] == checked["_ref_const_key"]
) & checked["url_match"]

print("Candidate rows:", len(candidates))
print("Rows after reference merge:", len(checked))
print("URL/year matches:", checked["url_match"].sum())
print("URL/year mismatches:", (~checked["url_match"]).sum())
print("State mismatches:", (checked["url_match"] & ~checked["state_match"]).sum())
print("Constituency mismatches:", (checked["url_match"] & ~checked["const_match"]).sum())

display(
    checked.loc[
        ~checked["url_match"] |
        ~checked["state_match"] |
        ~checked["const_match"],
        [
            "year", "state", "const", "sno", "candidate",
            "constituency_url", "ref_state", "ref_const", "ref_link"
        ]
    ].head(30)
)

Duplicate reference URL/year keys: 0
Candidate rows: 10395
Rows after reference merge: 10395
URL/year matches: 10395
URL/year mismatches: 0
State mismatches: 0
Constituency mismatches: 0


,year,state,const,sno,candidate,constituency_url,ref_state,ref_const,ref_link


In [36]:
asset_urls = candidates["asset_link"].dropna().astype(str).str.strip()
liability_urls = candidates["liability_link"].dropna().astype(str).str.strip()

all_urls = pd.Index(
    pd.concat(
        [asset_urls, liability_urls],
        ignore_index=True
    ).replace("", pd.NA).dropna().unique()
)

print("Unique asset/liability image URLs:", len(all_urls))

Unique asset/liability image URLs: 6520


In [37]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from pathlib import Path
import time

import pandas as pd
import requests
import pytesseract
from PIL import Image, ImageOps, ImageEnhance

# Keep the existing Tesseract installation path.
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

INTERIM_DIR = Path("../interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = INTERIM_DIR / "image_ocr_checkpoint.csv"

# Tune these two settings.
MAX_WORKERS = 8
BATCH_SIZE = 80

# Load previous results so successful URLs are not processed again.
if CHECKPOINT_PATH.exists():
    ocr_checkpoint = pd.read_csv(
        CHECKPOINT_PATH,
        dtype={"url": "string", "text": "string",
               "status": "string", "error": "string"}
    )
    ocr_checkpoint = ocr_checkpoint.drop_duplicates(
        subset="url", keep="last"
    )
else:
    ocr_checkpoint = pd.DataFrame(
        columns=["url", "text", "status", "error"]
    )

successful_urls = set(
    ocr_checkpoint.loc[
        ocr_checkpoint["status"].eq("ok"), "url"
    ].dropna().astype(str)
)

# all_urls should be the unique URL index created in your earlier cell.
urls_to_process = [
    str(url) for url in all_urls
    if pd.notna(url) and str(url).strip()
    and str(url) not in successful_urls
]

print("Already successful:", len(successful_urls))
print("URLs to process/retry:", len(urls_to_process))
print("Workers:", MAX_WORKERS, "| Batch size:", BATCH_SIZE)


def extract_image_text_fast(url):
    """Download and OCR one image; never save the image to disk."""
    response = None
    image = None
    processed = None

    try:
        response = requests.get(
            url,
            timeout=(10, 30),
            headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image.load()

        processed = ImageOps.grayscale(image)
        processed = ImageEnhance.Contrast(processed).enhance(2)

        text = pytesseract.image_to_string(
            processed,
            config="--oem 3 --psm 6"
        ).strip()

        return {
            "url": url,
            "text": text,
            "status": "ok" if text else "empty_text",
            "error": ""
        }

    except Exception as exc:
        return {
            "url": url,
            "text": "",
            "status": "error",
            "error": str(exc)[:500]
        }

    finally:
        if processed is not None:
            processed.close()
        if image is not None:
            image.close()
        if response is not None:
            response.close()


start_time = time.monotonic()

for batch_start in range(0, len(urls_to_process), BATCH_SIZE):
    batch = urls_to_process[batch_start:batch_start + BATCH_SIZE]
    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(extract_image_text_fast, url): url
            for url in batch
        }

        for future in as_completed(futures):
            results.append(future.result())

    # Merge new results into the checkpoint and save after each batch.
    ocr_checkpoint = pd.concat(
        [ocr_checkpoint, pd.DataFrame(results)],
        ignore_index=True
    ).drop_duplicates(subset="url", keep="last")

    ocr_checkpoint.to_csv(CHECKPOINT_PATH, index=False)

    done = min(batch_start + len(batch), len(urls_to_process))
    elapsed = time.monotonic() - start_time

    print(
        f"Processed {done}/{len(urls_to_process)} URLs this run; "
        f"batch size={len(results)}; "
        f"batch errors={sum(r['status'] == 'error' for r in results)}; "
        f"elapsed={elapsed / 60:.1f} min; checkpoint saved."
    )

print("OCR run complete.")
print("Checkpoint:", CHECKPOINT_PATH.resolve())


Already successful: 0
URLs to process/retry: 6520
Workers: 8 | Batch size: 80
Processed 80/6520 URLs this run; batch size=80; batch errors=0; elapsed=0.2 min; checkpoint saved.
Processed 160/6520 URLs this run; batch size=80; batch errors=0; elapsed=0.4 min; checkpoint saved.
Processed 240/6520 URLs this run; batch size=80; batch errors=0; elapsed=0.7 min; checkpoint saved.
Processed 320/6520 URLs this run; batch size=80; batch errors=0; elapsed=0.9 min; checkpoint saved.
Processed 400/6520 URLs this run; batch size=80; batch errors=0; elapsed=1.1 min; checkpoint saved.
Processed 480/6520 URLs this run; batch size=80; batch errors=0; elapsed=1.4 min; checkpoint saved.
Processed 560/6520 URLs this run; batch size=80; batch errors=0; elapsed=1.6 min; checkpoint saved.
Processed 640/6520 URLs this run; batch size=80; batch errors=0; elapsed=1.8 min; checkpoint saved.
Processed 720/6520 URLs this run; batch size=80; batch errors=0; elapsed=2.0 min; checkpoint saved.
Processed 800/6520 URLs

In [38]:
# Re-read the latest checkpoint from disk.
ocr_checkpoint = pd.read_csv(
    CHECKPOINT_PATH,
    dtype={"url": "string", "text": "string",
           "status": "string", "error": "string"}
).drop_duplicates(subset="url", keep="last")

url_to_text = ocr_checkpoint.set_index("url")["text"].to_dict()
url_to_status = ocr_checkpoint.set_index("url")["status"].to_dict()

processed = candidates.copy()

for col in ["asset_link", "liability_link"]:
    if col not in processed.columns:
        raise KeyError(f"Missing required column: {col}")

asset_urls = processed["asset_link"].astype("string").str.strip()
liability_urls = processed["liability_link"].astype("string").str.strip()

asset_ocr = asset_urls.map(url_to_text)
liability_ocr = liability_urls.map(url_to_text)

# Preserve any pre-existing text if the URL has no usable OCR result.
if "total_assets" not in processed.columns:
    processed["total_assets"] = pd.NA
if "liabilities" not in processed.columns:
    processed["liabilities"] = pd.NA

existing_assets = processed["total_assets"].astype("string")
existing_liabilities = processed["liabilities"].astype("string")

processed["total_assets"] = asset_ocr.where(
    asset_ocr.notna() & asset_ocr.str.strip().ne(""),
    existing_assets
)

processed["liabilities"] = liability_ocr.where(
    liability_ocr.notna() & liability_ocr.str.strip().ne(""),
    existing_liabilities
)

processed["asset_ocr_status"] = asset_urls.map(url_to_status)
processed["liability_ocr_status"] = liability_urls.map(url_to_status)

print("Candidate rows:", len(processed))
print("Asset URLs with successful OCR:",
      processed["asset_ocr_status"].eq("ok").sum())
print("Liability URLs with successful OCR:",
      processed["liability_ocr_status"].eq("ok").sum())
print("Asset URLs without successful OCR:",
      (asset_urls.notna() & ~processed["asset_ocr_status"].eq("ok")).sum())
print("Liability URLs without successful OCR:",
      (liability_urls.notna() & ~processed["liability_ocr_status"].eq("ok")).sum())


Candidate rows: 10395
Asset URLs with successful OCR: 3260
Liability URLs with successful OCR: 3260
Asset URLs without successful OCR: 0
Liability URLs without successful OCR: 0


In [39]:
from urllib.parse import urlparse, parse_qs
import re

def clean_id(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    # Handle IDs accidentally read as e.g. "3627.0".
    if re.fullmatch(r"\d+\.0", value):
        value = value[:-2]
    return value

def inspect_image_url(url, candidate_id, expected_col):
    url = "" if pd.isna(url) else str(url).strip()
    candidate_id = clean_id(candidate_id)

    result = {
        "url_present": bool(url),
        "valid_host": False,
        "endpoint_ok": False,
        "candidate_id_present": bool(candidate_id),
        "candidate_id_matches": False,
        "column_matches": False,
        "query_candidate_id": "",
        "query_col": "",
    }

    if not url:
        return result

    parsed = urlparse(url)
    query = parse_qs(parsed.query)
    query_id = query.get("candidate_id", [""])[0]
    query_col = query.get("col", [""])[0]

    result.update({
        "valid_host": parsed.hostname in {"myneta.info", "www.myneta.info"},
        "endpoint_ok": parsed.path.endswith("/image_v2.php"),
        "candidate_id_matches": (
            bool(candidate_id) and clean_id(query_id) == candidate_id
        ),
        "column_matches": query_col.lower() == expected_col,
        "query_candidate_id": query_id,
        "query_col": query_col,
    })
    return result


asset_checks = pd.DataFrame(
    [
        inspect_image_url(url, cid, "ta")
        for url, cid in zip(processed["asset_link"], processed["candidate_id"])
    ],
    index=processed.index
).add_prefix("asset_")

liability_checks = pd.DataFrame(
    [
        inspect_image_url(url, cid, "li")
        for url, cid in zip(processed["liability_link"], processed["candidate_id"])
    ],
    index=processed.index
).add_prefix("liability_")

processed = pd.concat(
    [processed, asset_checks, liability_checks], axis=1
)

print("Asset candidate-ID mismatches:",
      (processed["asset_url_present"] &
       ~processed["asset_candidate_id_matches"]).sum())

print("Liability candidate-ID mismatches:",
      (processed["liability_url_present"] &
       ~processed["liability_candidate_id_matches"]).sum())

print("Asset column mismatches:",
      (processed["asset_url_present"] &
       ~processed["asset_column_matches"]).sum())

print("Liability column mismatches:",
      (processed["liability_url_present"] &
       ~processed["liability_column_matches"]).sum())

display(processed.loc[
    (processed["asset_url_present"] &
     (~processed["asset_valid_host"] |
      ~processed["asset_endpoint_ok"] |
      ~processed["asset_candidate_id_matches"] |
      ~processed["asset_column_matches"]))
    |
    (processed["liability_url_present"] &
     (~processed["liability_valid_host"] |
      ~processed["liability_endpoint_ok"] |
      ~processed["liability_candidate_id_matches"] |
      ~processed["liability_column_matches"])),
    ["year", "state", "const", "candidate", "candidate_id",
     "asset_link", "liability_link"]
].head(30))


Asset candidate-ID mismatches: 0
Liability candidate-ID mismatches: 0
Asset column mismatches: 0
Liability column mismatches: 3260


,year,state,const,candidate,candidate_id,asset_link,liability_link
0,2024,BIHAR,ARARIA,Pradeep Kumar Singh,3627,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...
3,2024,BIHAR,ARARIA,Md. Ismail,4107,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...
8,2024,BIHAR,ARARIA,Zawed Akhtar,4775,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...
9,2024,BIHAR,ARRAH,Sudama Prasad,9539,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...
12,2024,BIHAR,ARRAH,Krishna Paswan,9546,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...
15,2024,BIHAR,ARRAH,Navnit Kumar,9547,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...
18,2024,BIHAR,ARRAH,Ramjee Singh,9540,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...
25,2024,BIHAR,AURANGABAD,Pratibha Rani,1703,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...
28,2024,BIHAR,AURANGABAD,Shailesh Rahi,1699,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...
31,2024,BIHAR,AURANGABAD,Sushil Kumar Singh,43,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...


In [40]:
def normalize_url(value):
    if pd.isna(value):
        return ""
    return str(value).strip().rstrip("/").casefold()

def normalize_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(value).casefold())

required_candidate_cols = [
    "year", "state", "const", "const_id", "constituency_url"
]
required_link_cols = [
    "year", "state_ut", "const", "const_id", "link"
]

missing_candidate_cols = [
    col for col in required_candidate_cols if col not in processed.columns
]
missing_link_cols = [
    col for col in required_link_cols if col not in links.columns
]

if missing_candidate_cols:
    raise KeyError(f"Candidate dataset missing: {missing_candidate_cols}")
if missing_link_cols:
    raise KeyError(f"Constituency reference dataset missing: {missing_link_cols}")

c = processed.copy()
l = links.copy()

c["_url_key"] = c["constituency_url"].map(normalize_url)
l["_url_key"] = l["link"].map(normalize_url)
c["_year_key"] = c["year"].map(normalize_text)
l["_year_key"] = l["year"].map(normalize_text)

# Identify duplicate reference keys before joining.
duplicate_reference_keys = l.duplicated(
    subset=["_url_key", "_year_key"], keep=False
)

print("Duplicate URL/year rows in constituency reference:",
      duplicate_reference_keys.sum())

display(l.loc[
    duplicate_reference_keys,
    ["year", "state_ut", "const", "const_id", "link"]
].sort_values(["year", "const"]).head(30))

# Only use unique reference keys for the join. If duplicates exist,
# they remain visible in the audit above.
l_unique = l.loc[
    ~l.duplicated(subset=["_url_key", "_year_key"], keep=False)
].copy()

reference = l_unique[
    ["_url_key", "_year_key", "state_ut", "const", "const_id", "link"]
].rename(columns={
    "state_ut": "ref_state",
    "const": "ref_const",
    "const_id": "ref_const_id",
    "link": "ref_link",
})

checked = c.merge(
    reference,
    on=["_url_key", "_year_key"],
    how="left",
    validate="many_to_one",
    indicator=True
)

checked["constituency_url_matched"] = checked["_merge"].eq("both")
checked["state_matches_reference"] = (
    checked["state"].map(normalize_text)
    == checked["ref_state"].map(normalize_text)
)
checked["constituency_id_matches_reference"] = (
    checked["const_id"].map(clean_id)
    == checked["ref_const_id"].map(clean_id)
)
checked["constituency_name_matches_reference"] = (
    checked["const"].map(normalize_text)
    == checked["ref_const"].map(normalize_text)
)

print("Candidate rows:", len(checked))
print("No matching constituency URL/year:",
      (~checked["constituency_url_matched"]).sum())
print("State mismatches among matched rows:",
      (checked["constituency_url_matched"] &
       ~checked["state_matches_reference"]).sum())
print("Constituency ID mismatches among matched rows:",
      (checked["constituency_url_matched"] &
       ~checked["constituency_id_matches_reference"]).sum())
print("Constituency name mismatches among matched rows:",
      (checked["constituency_url_matched"] &
       ~checked["constituency_name_matches_reference"]).sum())

display(checked.loc[
    ~checked["constituency_url_matched"]
    | (checked["constituency_url_matched"] &
       (~checked["state_matches_reference"]
        | ~checked["constituency_id_matches_reference"]
        | ~checked["constituency_name_matches_reference"])),
    ["year", "state", "const", "const_id", "candidate",
     "constituency_url", "ref_state", "ref_const",
     "ref_const_id", "ref_link"]
].head(50))


Duplicate URL/year rows in constituency reference: 0


,year,state_ut,const,const_id,link


Candidate rows: 10395
No matching constituency URL/year: 0
State mismatches among matched rows: 0
Constituency ID mismatches among matched rows: 0
Constituency name mismatches among matched rows: 0


,year,state,const,const_id,candidate,constituency_url,ref_state,ref_const,ref_const_id,ref_link


In [41]:
# The constituency merge must not increase or decrease the candidate row count.
assert len(checked) == len(processed), (
    f"Row count changed after reference merge: "
    f"{len(processed)} -> {len(checked)}"
)

# Candidate duplicates within the same year and constituency.
duplicate_key_cols = ["year", "const_id", "candidate_id"]
duplicates = checked.duplicated(
    subset=duplicate_key_cols, keep=False
) & checked["candidate_id"].notna()

print("Rows before saving:", len(checked))
print("Duplicate candidate/year/constituency rows:", duplicates.sum())
print("Missing candidate names:", checked["candidate"].isna().sum())
print("Missing candidate IDs:", checked["candidate_id"].isna().sum())
print("Missing constituency IDs:", checked["const_id"].isna().sum())
print("Missing constituency URLs:", checked["constituency_url"].isna().sum())

print("\nOCR status counts:")
display(ocr_checkpoint["status"].value_counts(dropna=False).to_frame("urls"))

print("\nRows with failed/empty OCR despite having an image URL:")
display(checked.loc[
    (
        checked["asset_url_present"]
        & ~checked["asset_ocr_status"].eq("ok")
    )
    | (
        checked["liability_url_present"]
        & ~checked["liability_ocr_status"].eq("ok")
    ),
    ["year", "state", "const", "candidate",
     "asset_link", "asset_ocr_status",
     "liability_link", "liability_ocr_status"]
].head(30))


Rows before saving: 10395
Duplicate candidate/year/constituency rows: 0
Missing candidate names: 0
Missing candidate IDs: 0
Missing constituency IDs: 0
Missing constituency URLs: 0

OCR status counts:


,urls
status,
ok,6520



Rows with failed/empty OCR despite having an image URL:


,year,state,const,candidate,asset_link,asset_ocr_status,liability_link,liability_ocr_status


In [42]:
INTERIM_DIR = Path("../interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# Save the full diagnostic dataset first; no candidate rows are dropped.
audit_path = INTERIM_DIR / "candidate_processing_audit.csv"
checked.to_csv(audit_path, index=False)

# Normalize the criminal-cases column name if needed.
if "criminal_cases" not in checked.columns and "criminal cases" in checked.columns:
    checked = checked.rename(columns={"criminal cases": "criminal_cases"})

final_columns = [
    "year",
    "state",
    "const",
    "sno",
    "candidate",
    "is_winner",
    "party",
    "criminal_cases",
    "education",
    "age",
    "total_assets",
    "liabilities",
    "is_byelection",
    "bye_election_date",
    "election_date",
]

# Preserve all rows; add missing expected columns as empty values.
for col in final_columns:
    if col not in checked.columns:
        checked[col] = pd.NA

final_candidates = checked[final_columns].copy()

final_path = INTERIM_DIR / "candidates_processed.csv"
final_candidates.to_csv(final_path, index=False)

print("Saved final candidates:", final_path.resolve())
print("Saved diagnostic audit:", audit_path.resolve())
print("Saved OCR checkpoint:", CHECKPOINT_PATH.resolve())
print("Final shape:", final_candidates.shape)
display(final_candidates.head())


Saved final candidates: D:\MyCoding\DA2402\DCV-Project\interim\candidates_processed.csv
Saved diagnostic audit: D:\MyCoding\DA2402\DCV-Project\interim\candidate_processing_audit.csv
Saved OCR checkpoint: D:\MyCoding\DA2402\DCV-Project\interim\image_ocr_checkpoint.csv
Final shape: (10395, 15)


,year,state,const,sno,candidate,is_winner,party,criminal_cases,education,age,total_assets,liabilities,is_byelection,bye_election_date,election_date
0,2024,BIHAR,ARARIA,6,Pradeep Kumar Singh,True,BJP,4,10th Pass,58,"Rs 1,60,59,806\n~1 Crore+.","Rs 16,79,230\n~16 Lacs+.",False,NaN,07-05-2024
1,2024,BIHAR,ARARIA,1,Akhilesh Kumar,False,IND,0,Doctorate,47,"Rs 2,47,11,288 ~ 2 Crore+","Rs 3,05,374 ~ 3 Lacs+",False,NaN,07-05-2024
2,2024,BIHAR,ARARIA,2,Md Ghousul Azam,False,BSP,0,10th Pass,38,"Rs 1,60,50,000 ~ 1 Crore+","Rs 14,00,000 ~ 14 Lacs+",False,NaN,07-05-2024
3,2024,BIHAR,ARARIA,3,Md. Ismail,False,Bharatiya Momin Front,0,Graduate Professional,56,"Rs 7,95,000\n~7 Lacs+.",Rs 0,False,NaN,07-05-2024
4,2024,BIHAR,ARARIA,4,Md. Mobinul Haque,False,IND,0,Post Graduate,51,"Rs 34,05,000 ~ 34 Lacs+",Rs 0 ~,False,NaN,07-05-2024
